# Margin: AI code review, step by step

Margin reviews code in two passes. Static analyzers (Ruff, Bandit, mypy, Radon, Vulture, Lizard,
detect-secrets, Opengrep) find concrete issues; then hosted LLMs judge those findings and add the
semantic ones no rule can express, each backed by the exact lines and code it points at. A router keeps
every call within the providers' free-tier limits.

This notebook installs Margin, scans a small project, reviews a file with LLMs, shows the report, and
recomputes the evaluation. Everything runs without API keys; with keys (see step 3) the review uses
real models.

Repository: https://github.com/kishorverse/code-review-assistant

## 1. Install

In [ ]:
!git clone --depth 1 https://github.com/kishorverse/code-review-assistant.git
%cd code-review-assistant/backend
!pip install --quiet uv
!uv sync --locked --quiet
!uv run margin --help

Opengrep adds Margin's own security rules. It is optional: without it, that analyzer is reported as
skipped and the scan continues.

In [ ]:
!curl --fail --silent --show-error --location --output /usr/local/bin/opengrep \
    https://github.com/opengrep/opengrep/releases/download/v1.30.0/opengrep_manylinux_x86
!echo "35779bdd72e92129c8df2a77f0c55e8c08356801ea92591ef32108d6b28d564c  /usr/local/bin/opengrep" | sha256sum --check  # public release checksum  # pragma: allowlist secret
!chmod +x /usr/local/bin/opengrep

## 2. Static analysis of a project

`eval/datasets/seeded/src` holds 32 small modules written for the evaluation: 22 with injected bugs,
vulnerabilities, performance and style issues, and 10 clean ones. Static analysis needs no keys.

In [ ]:
!uv run margin scan ../eval/datasets/seeded/src

## 3. Models (optional)

To review with real models, add your keys in Colab's **Secrets** panel (the key icon on the left):
`NVIDIA_API_KEY` from build.nvidia.com, `GEMINI_API_KEY` from Google AI Studio, or `HF_TOKEN` from
Hugging Face. Without any key, Margin's mock provider answers instead, so every step still runs.
Code is only sent to hosted models with `--allow-external`, after detected secrets are masked.

In [ ]:
import os

MODELS = {
    "NVIDIA_API_KEY": ("NVIDIA_MODEL", "nvidia/nemotron-3-super-120b-a12b"),
    "GEMINI_API_KEY": ("GEMINI_MODEL", "gemini-3.5-flash"),
    "HF_TOKEN": ("HF_MODEL_LARGE", "openai/gpt-oss-120b"),
}
try:
    from google.colab import userdata
except ImportError:  # not running in Colab
    userdata = None


def read_secret(name):
    # A key from Colab Secrets, or from the environment outside Colab.
    if userdata is None:
        return os.environ.get(name)
    try:
        return userdata.get(name)
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        return None


found = []
for key, (model_setting, model) in MODELS.items():
    value = read_secret(key)
    if value:
        os.environ[key] = value
        os.environ.setdefault(model_setting, model)
        found.append(key)

if found:
    print("Reviewing with:", ", ".join(found))
else:
    os.environ["LLM_MODE"] = "mock"
    print("No keys found: using the mock provider, which gives canned answers.")

## 4. Review a file with LLMs

`--depth standard` reviews every chunk for bugs, security, performance and style, and has a second
model cross-check serious AI findings. The HTML report is self-contained.

In [ ]:
!uv run margin scan ../eval/datasets/seeded/src/billing.py --depth standard --allow-external \
    --format html --output /content/report.html
!uv run margin scan ../eval/datasets/seeded/src/billing.py --depth standard --allow-external \
    --format json --output /content/report.json

In [ ]:
import json
from pathlib import Path

import pandas as pd

report = json.loads(Path("/content/report.json").read_text(encoding="utf-8"))
print(f"Quality score: {report['score']['score']} ({report['score']['grade']})")
pd.DataFrame(
    [
        {
            "line": f["start_line"],
            "severity": f["severity"],
            "category": f["category"],
            "title": f["title"],
            "found by": ", ".join(f["sources"]),
            "status": f["status"],
        }
        for f in report["findings"]
    ]
)

In [ ]:
from pathlib import Path

from IPython.display import HTML

HTML(Path("/content/report.html").read_text(encoding="utf-8"))

## 5. Evaluation

The evaluation's runs are committed, so the metrics can be recomputed without keys: precision, recall
and F1 per configuration (static only, LLM only, hybrid, hybrid with cross-model verification) and per
model. The full report is `docs/evaluation.md`.

In [ ]:
!uv run python -m evaluation score
from pathlib import Path

from IPython.display import Markdown

Markdown(Path("../eval/results/tables.md").read_text(encoding="utf-8"))

The router simulation replays a scan's model calls against simulated providers with per-minute limits,
comparing a naive client with Margin's router. It runs in seconds and needs no keys.

In [ ]:
!uv run python -m evaluation routing

## 6. The web interface

The web UI (upload, live progress, and a results workspace with the code, its findings and the reviewer's
decisions) runs locally:

```bash
cd backend && uv run uvicorn app.main:create_app --factory
cd frontend && npm ci && npm run dev    # then open http://localhost:5173
```

See the README for details.